# MVC / MVT фреймворки на примере Django


## План урока

1. Классическая структура веб‑сервиса и Tiered architecture
2. Паттерн MVC vs MVT — сравнение и соответствие в Django
3. Установка и создание проекта Django
4. Как устроен Django (файловая структура, request lifecycle, middleware, ORM)
5. Конфигурирование и эксплуатация (settings, env, logging, static/media, миграции)
6. Лучшие практики развертывания (Gunicorn, Daphne/uvicorn, Nginx, Docker, systemd)
7. Асинхронные возможности Django (async views, ASGI, Channels, DB и ограничения)
8. Полезные команды, чеклист перед деплоем и дополнительные материалы

---


## 1. Классическая структура веб‑сервисов

**Layered / Tiered architecture** — популярный способ организации систем:

- **Presentation (UI)** — браузер / мобильное приложение; отвечает за отображение и UX.
- **Application / Service** — логика приложения (API, контроллеры, validation, workflows).
- **Domain / Business logic** — правила предметной области, модели, сервисы.
- **Data / Persistence** — базы данных, кэши, файловое хранилище.

Типичная диаграмма: Browser ⇄ Web Server (Nginx) ⇄ App Server (Django WSGI/ASGI) ⇄ Database / Cache / External APIs.
<img src="Screenshot1.png" width="800"/>
Преимущества: разделение ответственности, масштабирование по слоям, более простое тестирование.

---

## 2. Паттерн MVC vs MVT

**MVC (Model-View-Controller)**:
- Model — данные и правила работы с ними.
- View — представление данных (UI).
- Controller — принимает input, обновляет model и view.

**MVT (Model-View-Template)** — термин, который часто используют в Django:
- Model — как и в MVC, ORM‑модели.
- View — в Django это *логика*, которая получает запрос и возвращает ответ (аналог контроллера в MVC).
- Template — шаблон визуального представления (HTML).
<br>
<img src="Screenshot2.png" width="400"/>
<br>
**Как сопоставляется в Django:**
- Django View == Controller в классическом MVC.
- Template == View в MVC (слой отображения).
- Model == Model.

**Пример:** Django view получает `request`, использует модели (Model) и рендерит Template.

---

API -> DRF 

## 3. Установка Django и быстрый старт

Рекомендуется использовать виртуальное окружение.

```bash
python -m venv .venv
source .venv/bin/activate   # macOS / Linux
.\.venv\Scripts\activate  # Windows
pip install --upgrade pip
pip install django
```

Создание проекта и приложения:

```bash
django-admin startproject mysite
cd mysite
python manage.py startapp main
python manage.py migrate
python manage.py runserver
```

Создаём superuser:

```bash
python manage.py createsuperuser
```

---

## 4. Как устроен Django: структуры и lifecycle

**Файловая структура (корень проекта)**

- `manage.py` — CLI-обёртка для управления проектом.
- `mysite/` — пакет проекта, содержит `settings.py`, `urls.py`, `asgi.py`, `wsgi.py`.
- `apps/` — приложения, например `main/` с `models.py`, `views.py`, `templates/`, `urls.py`.

**Admin panel**

**Request lifecycle (упрощённо):**
1. Веб‑сервер направляет запрос в приложение (WSGI/ASGI entrypoint).
2. Django разбирает URL (URLconf в `urls.py`).
3. Middleware выполняются (request phase).
4. Вызывается view (sync или async). View использует ORM, Forms и т.д.
5. Формируется Response (шаблон, JSON) и проходит middleware (response phase).
6. Ответ возвращается клиенту.

**Middleware** — цепочка, которая может обрабатывать request/response, добавлять заголовки, авторизацию, сессии и т.д.

**ORM** — мощный слой работы с базой; модели описываются как классы Python.

---

In [ ]:
# Пример простого приложения: модель, view и шаблон (файлы упрощённо)

# models.py
from django.db import models

class Article(models.Model):
    title = models.CharField(max_length=200)
    body = models.TextField()
    created = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.title

# views.py (sync view)
from django.shortcuts import render
from .models import Article

def index(request):
    qs = Article.objects.order_by('-created')[:10]
    return render(request, 'main/index.html', {'articles': qs})

API
djangorestframework

# template: templates/main/index.html
{% for a in articles %}
  <h2>{{ a.title }}</h2>
  <p>{{ a.body|truncatechars:200 }}</p>
{% endfor %}

select_related; JOIN
prefetch_related;

.count() и .exists()

## 5. Конфигурирование и эксплуатация

**Settings: разделение и секреты**
- Не храните секреты в `settings.py`.
- Используйте переменные окружения (`os.environ`) или пакеты типа `django-environ`.
- Разделяйте `settings` на `base.py`, `development.py`, `production.py`.

**Static & Media**
- `STATIC_ROOT` + `collectstatic` для production.
- `MEDIA_ROOT` для загружаемых пользователем файлов; храните в S3/MinIO для масштабирования.

**Миграции**
- `python manage.py makemigrations` и `migrate`.
- Отслеживайте миграции в VCS.

**Логи**
- Настройте `LOGGING` в settings; отправляйте важные логи в централизованные системы (Sentry, ELK).

**Мониторинг и health checks**
- Простые `/health` views, метрики Prometheus, трассировка (OpenTelemetry).

---

## 6. Лучшие практики развертывания Django-проектов

**WSGI vs ASGI**
- WSGI (Gunicorn, uWSGI) — для синхронных приложений.
- ASGI (Daphne, Uvicorn, Hypercorn) — поддерживает async, WebSockets.

**Recommended stack (classic)**
- Nginx (reverse proxy, static serving) + Gunicorn (WSGI workers) + PostgreSQL + Redis (cache, celery broker).

**Async stack (если нужен async / websockets)**
- Nginx -> Daphne/Uvicorn -> Django ASGI -> Channels (если надо)

**Systemd unit (пример для Gunicorn)**
```
[Unit]
Description=gunicorn daemon
After=network.target

[Service]
User=www-data
Group=www-data
WorkingDirectory=/srv/mysite
ExecStart=/srv/mysite/.venv/bin/gunicorn mysite.wsgi:application \
    --workers 3 --bind unix:/run/gunicorn.sock

[Install]
WantedBy=multi-user.target
```

**Nginx snippet**
```
server {
    listen 80;
    server_name example.com;

    location = /favicon.ico { access_log off; log_not_found off; }
    location /static/ {
        root /srv/mysite/;
    }

    location / {
        include proxy_params;
        proxy_pass http://unix:/run/gunicorn.sock;
    }
}
```

**Docker + Docker Compose** — рекомендую контейнеризировать приложение, DB и Redis, но следите за персистентностью данных и секретами (use secrets, env files, external volumes).

**Security checklist**
- DEBUG=False
- SECURE_PROXY_SSL_HEADER set если за прокси
- SESSION_COOKIE_SECURE, CSRF_COOKIE_SECURE
- HSTS
- use allowed hosts
- keep dependencies up to date

---

In [ ]:
# Dockerfile (пример для Django)
# Используйте официальный Python image
FROM python:3.12-slim
WORKDIR /app
ENV PYTHONDONTWRITEBYTECODE 1
ENV PYTHONUNBUFFERED 1

RUN apt-get update && apt-get install -y build-essential libpq-dev && rm -rf /var/lib/apt/lists/*
COPY pyproject.toml poetry.lock* /app/
# Если используете pip:
COPY requirements.txt /app/
RUN pip install --upgrade pip
RUN pip install -r requirements.txt

COPY . /app/
CMD ["gunicorn", "mysite.wsgi:application", "--bind", "0.0.0.0:8000"]

# docker-compose.yml (упрощённо)
# version: '3.8'
# services:
#   web:
#     build: .
#     command: gunicorn mysite.wsgi:application --bind 0.0.0.0:8000
#     volumes:
#       - .:/app
#     env_file: .env
#     depends_on:
#       - db
#   db:
#     image: postgres:16
#     volumes:
#       - pgdata:/var/lib/postgresql/data
# volumes:
#   pgdata:


## 7. Асинхронные возможности Django

**ASGI support**
- Начиная с Django 3.0 проект поддерживает ASGI.
- В `asgi.py` создаётся ASGI application; серверы — Daphne/Uvicorn.

**Async views**
- Можно писать `async def my_view(request):` и использовать `await` внутри.
- **Важно:** большинство ORM операций (до недавних версий) были синхронными; современные Django версии предоставляют экспериментальную/постепенную async поддержку для ORM, но зачастую нужно вызывать `database_sync_to_async` или использовать синхронный режим в async view для доступа к ORM.

**Пример async view**
```python
# views.py
from django.http import JsonResponse

async def ping(request):
    await some_async_io()  # например, вызов внешнего API через httpx
    return JsonResponse({'ok': True})
```

**Django Channels**
- Для WebSocket, background tasks, long‑running connections. Channels добавляет слой routing и consumers.
- Работает с ASGI и использует слой канала (Channel layer) — чаще Redis.

**Асинхронность и DB**
- ORM historically is synchronous — calling it in async context may block event loop.
- Use `asgiref.sync.sync_to_async` / `database_sync_to_async` when calling ORM from async code, or rely on database drivers supporting async (e.g., asyncpg + libraries).

**Celery vs Async**
- Celery для фоновых задач (своё масштабирование и управление). Async view не заменяет очередь задач.

---

In [ ]:
# Пример использования ORM из async-view
from asgiref.sync import sync_to_async
from django.http import JsonResponse
from .models import Article

@sync_to_async
def get_latest_titles():
    return list(Article.objects.order_by('-created').values_list('title', flat=True)[:10])

async def latest_titles(request):
    titles = await get_latest_titles()
    return JsonResponse({'titles': titles})


## 8. Полезные команды и чеклист перед деплоем

**Команды**
- `python manage.py check --deploy` — рекомендации для продакшна.
- `python manage.py makemigrations` / `migrate`.
- `python manage.py collectstatic`.

**Чеклист перед деплоем**
- DEBUG=False, ALLOWED_HOSTS настроен.
- Секреты вынесены в env.
- HTTPS настроен, HSTS включён.
- Логи и мониторинг подключены.
- Миграции выполнены и протестированы на staging.
- Резервное копирование БД настроено.

---

## Дополнительно и источники для самостоятельного чтения

- Официальная документация Django: https://docs.djangoproject.com/
- Django Channels: https://channels.readthedocs.io/
